# Imports

In [1]:
import numpy as np
from tensorflow.keras.datasets import mnist
from sklearn.decomposition import PCA

from methods.bcgd_gauss_southwell import gs_bcgd
from methods.bcgd_randomized import randomized_bcgd
from methods.gradient_descent import gradient_descent
from methods.utils.accuracy import accuracy
from methods.utils.block_size import block_step_size
from methods.utils.gradient import MultiLogisticRegressionOracle, CachedMultiLogisticRegressionOracle

rng = np.random.default_rng(seed=42) # Turnoff or modify if no seeding will be used.

I0000 00:00:1777284278.428925  539248 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777284278.524923  539248 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777284280.382945  539248 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


# Data Loading

## Synthetic Data

In [2]:
n_categories = 50
n_features = 500
n_examples = 1000

syntetic_features = rng.random((n_examples, n_features))
X = rng.random((n_features, n_categories))
E = rng.random((n_examples, n_categories))

transformation = np.dot(syntetic_features,X) + E
syntetic_labels = np.argmax(transformation, axis= 1)

syntetic_data = {
    "n_categories": n_categories,
    "n_features": n_features,
    "train_features": syntetic_features,
    "train_labels": syntetic_labels,
    "test_features": ...,
    "test_labels": ...,
}

## MNIST Dataset

In [3]:
n_components = 50
(mnist_features, mnist_labels), (mnist_features_test, mnist_labels_test) = mnist.load_data()

mnist_features = mnist_features.reshape((mnist_features.shape[0],-1))
mnist_features_test = mnist_features_test.reshape((mnist_features_test.shape[0],-1))

pca = PCA(n_components=n_components) # Keep 2 components
mnist_reduced_features = pca.fit_transform(mnist_features)
mnist_reduced_features_test = pca.transform(mnist_features_test)

mnist_data = {
    "n_categories": 10,
    "n_features": n_components,
    "train_features": mnist_reduced_features,
    "train_labels": mnist_labels,
    "test_features": mnist_reduced_features_test,
    "test_labels": mnist_labels_test,
}

## Bloodcell Dataset

(60000, 784)

# Method Calls
This section is for implementing all the methods that will be used for generating graphs or any other types of solutions

In [4]:
active_data = mnist_data

number_of_categories = 10
number_of_features = mnist_reduced_features.shape[1]

#calculated step sizes
step_sizes = block_step_size(active_data["train_features"])

oracle = MultiLogisticRegressionOracle(active_data["train_features"], active_data["train_labels"], active_data["n_categories"])
cached_oracle = CachedMultiLogisticRegressionOracle(active_data["train_features"], active_data["train_labels"], number_of_categories)

## Gradient Descent

In [ ]:
gradient_descent_results = gradient_descent(
    np.zeros((active_data["n_features"], active_data["n_categories"])),
    oracle,
    0.001,  # Reduced step size to prevent NaN
    tol=1e-5,
    debug=True,
)

/home/jjb_1/master/second_semester/optimization/assignment_optimization_2025_2026/methods/utils/gradient.py:28: RuntimeWarning: overflow encountered in exp
  transformed_entries = np.exp(np.dot(self.predictors, x)) # m x k
/home/jjb_1/master/second_semester/optimization/assignment_optimization_2025_2026/methods/utils/gradient.py:30: RuntimeWarning: invalid value encountered in divide
  softmax_entries = transformed_entries / summed_entries # m x k


Iteration 500: Loss = nan, Accuracy = 0.09871666666666666


KeyboardInterrupt: 

## BCGD Randomized

In [6]:
randomized_bcgd_results = randomized_bcgd(
    np.zeros((active_data["n_features"], active_data["n_categories"])),
    oracle,
    step_sizes,
    tol=1.4e-05,
    debug=True,
)

iteration: 0, sequential distance: 0.0020151392723768906, loss: 130001.5614462594
accuracy: 0.17148333333333332
iteration: 500, sequential distance: 0.00011693927354291781, loss: 28292.30023575941
accuracy: 0.8752833333333333


In [7]:
accuracy(active_data["test_features"],active_data["test_labels"], randomized_bcgd_results["x_k"])

np.float64(0.8892)

## Gauss-Southwell BCGD

In [8]:
gs_bcgd_results = gs_bcgd(
    np.zeros((active_data["n_features"], active_data["n_categories"])),
    cached_oracle,
    step_sizes,
    tol=0.00001,
    debug=True,
)

iteration: 500, selected_block: 5, sequential distance: 4.083378509974746e-05, loss: 25733.303154
accuracy: 87.85%


In [9]:
accuracy(active_data["test_features"],active_data["test_labels"], gs_bcgd_results["x_k"])

np.float64(0.896)